# 🚀 Fabrika GPU İşçisi — Kaggle / Colab (FLUX.2 + Wan2.1 VACE + Suno Seviyesi ACE-Step 1.5 Müzik)

Bu notebook, `yusasufhin.com` Video ve Müzik stüdyolarına **gerçek yerel AI difüzyon** yapan ücretsiz bir GPU
işçisi bağlar. Sahte ffmpeg kutusu veya bip sesi üretmez; `gpu_worker.py` çalıştırır.

- **Tutarlı Karakter & Video (Higgsfield Seviyesi):** FLUX.2-klein-4B (çoklu referans portre + seed kilidi), Z-Image-Turbo, **Wan2.1-VACE-1.3B (Referans→Video R2V & İlk+Son Kare FLF2V)**, Wan2.1-I2V-14B, Wan2.2-TI2V-5B, LTX-Video
- **Ücretsiz AI Müzik (Suno Seviyesi):** **ACE-Step 1.5 XL Turbo** (`ACE-Step/acestep-v15-xl-turbo-diffusers`, MIT lisanslı, 48 kHz gerçek stereo, Türkçe/çok dilli vokal + `[Verse]/[Chorus]` şarkı sözü + BPM/Gam + Cover/Repaint/Extend, 4 dk'ya kadar!) + **Meta MusicGen Stereo Medium** (hızlı enstrümantal yedek)

## Adımlar
1. **Kaggle:** `Settings → Accelerator → GPU T4 x2` · **Colab:** `Runtime → Change runtime type → T4 GPU`.
2. İşçi anahtarını `FABRIKA_WORKER_KEY` olarak tanımlayın:
   - Kaggle: `Add-ons → Secrets` içine `FABRIKA_WORKER_KEY` ekleyin.
   - Colab: 🔑 (Secrets) panelinden `FABRIKA_WORKER_KEY` ekleyin.
   - Ya da hücre çalışırken `getpass` soracaktır.
3. Hücreleri sırayla çalıştırın. İşçi hem Video (`vace_r2v`, `flf2v`, `wan_i2v`, `flux2_resim`) hem de Müzik (`gpu-acestep-1.5`, `gpu-musicgen`) işlerini otomatik alır, üretir ve geri yükler.

> **VRAM dürüstlüğü:** T4 15–16 GB'da FLUX.2-klein-4B, Wan2.1-VACE-1.3B, Wan2.1-T2V-1.3B, **ACE-Step 1.5 XL Turbo** (CPU offload + VAE tiling ~6–8 GB VRAM) ve MusicGen Stereo rahatça sığar.
> Wan2.1-I2V-14B, `bitsandbytes` NF4 4-bit kuantizasyonla sığar; sığmazsa işçi dürüst hata bildirir, sahte çıktı üretmez.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/sufhinyusa/yusasufhin-cockpit.git"
if not Path("gpu_worker.py").is_file():
    if not Path("yusasufhin-cockpit").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("yusasufhin-cockpit")

# GPU bağımlılıkları (torch/diffusers/transformers/soundfile/bitsandbytes)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt"], check=True)
# AceStepPipeline + Flux2KleinPipeline + WanVACEPipeline güvencesi (diffusers main)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/diffusers.git"], check=False)
print("Çalışma dizini:", Path.cwd())
print("gpu_worker.py:", Path("gpu_worker.py").is_file())


In [ ]:
import os
from getpass import getpass

WORKER_KEY = (
    os.environ.get("FABRIKA_WORKER_KEY")
    or os.environ.get("WORKER_KEY")
    or os.environ.get("FABRIKA_TOKEN")
)

if not WORKER_KEY:
    try:  # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        WORKER_KEY = UserSecretsClient().get_secret("FABRIKA_WORKER_KEY")
    except Exception:
        pass

if not WORKER_KEY:
    try:  # Colab Secrets
        from google.colab import userdata
        WORKER_KEY = userdata.get("FABRIKA_WORKER_KEY")
    except Exception:
        pass

if not WORKER_KEY:
    WORKER_KEY = getpass("FABRIKA_WORKER_KEY: ").strip()

if not WORKER_KEY:
    raise SystemExit("FABRIKA_WORKER_KEY yok — işçi kimlik doğrulaması yapılamaz.")

os.environ["FABRIKA_WORKER_KEY"] = WORKER_KEY
os.environ.setdefault("FABRIKA_SUNUCU", "https://yusasufhin.com")
print("Anahtar: ayarlı (Bearer + X-Worker-Key)")
print("Sunucu:", os.environ["FABRIKA_SUNUCU"])


In [ ]:
# GPU + model kullanılabilirliğini raporla (iş almadan çıkar)
import os, subprocess, sys

subprocess.run([
    sys.executable, "gpu_worker.py",
    "--server", os.environ["FABRIKA_SUNUCU"],
    "--token", os.environ["FABRIKA_WORKER_KEY"],
    "--gpu-kontrol",
], check=False)


In [ ]:
# Gerçek işçi döngüsü — Ctrl/Cmd+C ile durdurun
import os, subprocess, sys

subprocess.run([
    sys.executable, "gpu_worker.py",
    "--server", os.environ["FABRIKA_SUNUCU"],
    "--token", os.environ["FABRIKA_WORKER_KEY"],
    "--kalite", "480p",
    "--modlar", "all",
    "--muzik-model", "medium",
], check=False)
